# Traffic Signal OpenEnv: LLM-Based Reinforcement Learning Pipeline

This notebook implements a hardened RL fine-tuning pipeline for hierarchical traffic control.

In [ ]:
!pip -q uninstall -y trl transformers llm-blender mergekit tokenizers
!pip -q install --no-cache-dir \
    "transformers==4.44.2" \
    "trl==0.19.1" \
    "llm-blender==0.0.2" \
    "mergekit>=0.0.4" \
    "tokenizers<0.20" \
    unsloth wandb matplotlib requests numpy datasets accelerate peft bitsandbytes
print("Install complete. Restart runtime once, then continue.")


In [ ]:
import os
import gc
import json
import re
import time
import random
import requests
import numpy as np
import torch
import wandb
from datasets import Dataset
from unsloth import FastLanguageModel, PatchFastRL
try:
    from trl import GRPOTrainer, GRPOConfig
except Exception as exc:
    raise RuntimeError("Missing TRL deps. Run: !pip install -q mergekit llm-blender trl") from exc


In [ ]:
PatchFastRL("GRPO", FastLanguageModel)
print("PatchFastRL applied.")


In [ ]:
ENV_URL = "https://guuru-dev-traffic-signal-openenv-2.hf.space"
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
print("ENV_URL:", ENV_URL)

try:
    wandb.init(project="traffic-signal-rl", name="full-run-1")
    print("WandB initialized.")
except Exception as e:
    print(f"WandB init failed (non-fatal): {e}")
    wandb = None

r = requests.get(f"{ENV_URL}/health", timeout=30)
assert r.status_code == 200, f"Space not healthy: {r.status_code} {r.text}"
print("Space status:", r.json()["status"])
print("Task count:", r.json()["task_count"])
print("Ready to load model.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seeds set to {SEED}.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.generation_config.do_sample = True
model.generation_config.temperature = 0.7
model.generation_config.top_p = 0.9
print("Model loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
PROMPT_BANK = [
    'Control traffic at a 4-way intersection. Output JSON with local_actions and central_action.',
    'Minimize congestion and waiting time. Return valid JSON actions.',
    'Optimize traffic flow under heavy load conditions.',
    'Handle emergency vehicles with priority while keeping flow balanced.',
    'Reduce queue lengths across all intersections.',
    'Prevent spillback and gridlock using coordinated actions.',
    'Balance throughput and fairness across directions.',
    'Respond to dynamic traffic demand changes efficiently.',
    'Avoid starvation of any direction while optimizing flow.',
    'Maximize throughput without increasing waiting time.',
    'Handle congestion recovery after traffic spikes.',
    'Ensure smooth transitions between traffic phases.',
    'Use central coordination to improve global performance.',
    'Prioritize critical lanes under pressure.',
    'Optimize intersection switching intelligently.',
    'Maintain stability while improving efficiency.',
    'Avoid oscillatory switching behavior.',
    'Keep traffic flowing evenly across all directions.',
    'Adapt to changing traffic density dynamically.',
    'Prevent long queues from forming.',
]

train_prompts = PROMPT_BANK * 2
train_dataset = Dataset.from_dict({"prompt": train_prompts})
print("Dataset size:", len(train_dataset))

def safe_post(url, action, retries=16, timeout=60):
    for attempt in range(retries):
        try:
            r = requests.post(url, json=action, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                wait = min(45, 2 * (attempt + 1)) + random.uniform(0.0, 1.0)
                print(
                    f"HTTP {r.status_code}. Waiting {wait:.1f}s "
                    f"(attempt {attempt+1}/{retries})"
                )
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            wait = min(30, 2 + attempt)
            print(
                f"Network/timeout error on attempt {attempt+1}/{retries}. "
                f"Retrying in {wait}s..."
            )
            time.sleep(wait)
    raise RuntimeError(f"Failed after {retries} retries: {url}")

def parse_action(completion):
    valid = {"KEEP", "SWITCH", "PHASE_0", "PHASE_1", "PHASE_2", "PHASE_3"}
    base = {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"}

    try:
        action = json.loads(completion)
        if isinstance(action, dict):
            raw_local = action.get("local_actions", {})
            clean_local = {}
            for key in ("NW", "NE", "SW", "SE"):
                raw_val = str(raw_local.get(key, "KEEP")).upper().strip() if isinstance(raw_local, dict) else "KEEP"
                clean_local[key] = raw_val if raw_val in valid else "KEEP"
            return {"local_actions": clean_local, "central_action": {}}, False
    except Exception:
        pass

    try:
        match = re.search(r'\{.*\}', completion, re.DOTALL)
        if match:
            parsed_action, _ = parse_action(match.group())
            return parsed_action, True
    except Exception:
        pass

    return {"local_actions": base, "central_action": {}}, True

# Anti-reward-hacking properties:
# 1. Deterministic seeded transitions
# 2. 6 independent rubric components
# 3. Reward clipped to [-1.0, 1.0]
# 4. total_priority_budget constraint prevents all-boost exploitation
# 5. Episode-level final_score prevents short-term gaming
def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for episode, (prompt, completion) in enumerate(zip(prompts, completions), start=1):

        task_id = "medium_dynamic" if episode < 20 else "hard_multi"
        safe_post(f"{ENV_URL}/reset", {"task_id": task_id, "central_enabled": True})

        action, is_hallucinated = parse_action(completion)

        episode_reward = 0.0
        done = False
        step_count = 0
        info = {}
        latency_ms = 0.0

        while not done and step_count < 100:
            t0 = time.time()
            try:
                step_res = safe_post(f"{ENV_URL}/step", action)
            except requests.HTTPError as exc:
                if getattr(exc.response, "status_code", None) == 422:
                    action = {
                        "local_actions": {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"},
                        "central_action": {},
                    }
                    step_res = safe_post(f"{ENV_URL}/step", action)
                else:
                    raise
            latency_ms = (time.time() - t0) * 1000
            data = step_res.json()
            info = data.get("info", {})
            episode_reward += float(data.get("reward", 0.0))
            done = data.get("done", False)
            step_count += 1
            time.sleep(0.05)

        if is_hallucinated:
            episode_reward -= 5.0  # Heavy penalty for invalid JSON to prevent safe fallback exploitation

        log_data = {
            "episode_reward": episode_reward,
            "mean_queue": info.get("mean_queue", 0.0),
            "final_score": info.get("final_score", 0.0),
            "throughput": info.get("throughput", 0.0),
            "step_count": step_count,
            "step_latency_ms": latency_ms,
            "is_hallucinated": float(is_hallucinated),
        }
        if wandb:
            wandb.log(log_data)

        if episode % 5 == 0:
            print(f"[DEBUG] reward={episode_reward:.4f} | parsed_action={action}")

        if episode % 2 == 0:
            print(f"\n=== Episode {episode} ===")
            print(f"  Reward   : {episode_reward:.4f}")
            print(f"  Score    : {info.get('final_score', 'N/A')}")
            print(f"  Queue    : {info.get('mean_queue', 'N/A')}")
            print(f"  Steps    : {step_count}")
            print(f"  Latency  : {latency_ms:.1f}ms")
            print(f"  Action   : {action}")
            # WARNING: if action is always identical across episodes,
            # reward hacking may be occurring. Stop and inspect.

        rewards.append(episode_reward)
        time.sleep(0.2)

    return rewards

print("Reward function ready.")

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision config -> bf16={use_bf16}, fp16={use_fp16}")

training_args = GRPOConfig(
    output_dir="./outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    max_steps=80,
    max_prompt_length=512,
    max_completion_length=64,
    num_generations=4,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="wandb",
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    logging_steps=1,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
torch.cuda.empty_cache()
gc.collect()
print("Trainer ready. Launching training...")

checkpoint_paths = []
if os.path.exists("./outputs"):
    checkpoint_paths = [
        os.path.join("./outputs", name)
        for name in os.listdir("./outputs")
        if name.startswith("checkpoint-") and os.path.isdir(os.path.join("./outputs", name))
    ]
checkpoint_paths = sorted(checkpoint_paths, key=lambda path: int(path.rsplit("-", 1)[-1]))

if checkpoint_paths:
    latest_checkpoint = checkpoint_paths[-1]
    print("Resuming from:", latest_checkpoint)
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("Starting fresh run")
    trainer.train()

print("\nTraining complete.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# NEVER use merge-and-unload on a 4-bit model — corrupts weights
model.save_pretrained("outputs/traffic-lora")
tokenizer.save_pretrained("outputs/traffic-lora")
print("Adapter weights saved to outputs/traffic-lora")
# Optional push:
# model.push_to_hub("Guuru-DEV/traffic-signal-agent", token=os.getenv("HF_TOKEN"))
# tokenizer.push_to_hub("Guuru-DEV/traffic-signal-agent", token=os.getenv("HF_TOKEN"))


In [ ]:
if wandb:
    wandb.finish()
print("WandB run complete.")


In [ ]:
# ============================================================
# AUTO-SAVE RESULTS AFTER TRAINING
# Pushes model and plots to HF Hub only
# Clone locally later and push to GitHub manually
# ============================================================
import os
import sys
from huggingface_hub import HfApi

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Add it as a Space Secret or run: export HF_TOKEN=your_token")

api = HfApi(token=HF_TOKEN)

# Step 1: Verify model weights exist
assert os.path.exists("outputs/traffic-lora"),     "FAIL: outputs/traffic-lora not found. Training may not have completed."
print("Model weights found.")
print("Contents:", os.listdir("outputs/traffic-lora"))

# Step 2: Push model adapter weights to HF Hub model repo
api.create_repo(
    "Guuru-DEV/traffic-signal-agent",
    repo_type="model",
    exist_ok=True
)
api.upload_folder(
    folder_path="outputs/traffic-lora",
    repo_id="Guuru-DEV/traffic-signal-agent",
    repo_type="model"
)
print("Model saved to: https://huggingface.co/Guuru-DEV/traffic-signal-agent")

# Step 3: Generate plots
os.makedirs("plots", exist_ok=True)
try:
    sys.path.insert(0, ".")
    from env.metrics_exporter import generate_training_plots

    # 🔴 FIX: ensure collected_data exists before using it
    if "collected_data" not in globals():
        raise RuntimeError("collected_data not found. Ensure training loop stores episode logs.")

    generate_training_plots(collected_data, "plots")
    print("Plots generated in plots/")
except Exception as e:
    print(f"Plot generation skipped (non-fatal): {e}")
    print("Download WandB screenshots manually instead.")

# Step 4: Push plots to HF Hub dataset repo for safe storage
try:
    # 🔴 FIX: only upload if plots were actually created
    if os.path.exists("plots") and len(os.listdir("plots")) > 0:
        api.create_repo(
            "Guuru-DEV/traffic-signal-plots",
            repo_type="dataset",
            exist_ok=True
        )
        api.upload_folder(
            folder_path="plots",
            repo_id="Guuru-DEV/traffic-signal-plots",
            repo_type="dataset"
        )
        print("Plots saved to: https://huggingface.co/datasets/Guuru-DEV/traffic-signal-plots")
    else:
        print("No plots found — skipping upload.")
except Exception as e:
    print(f"Plot upload skipped (non-fatal): {e}")

# Step 5: Print final summary
print("\n" + "="*55)
print("TRAINING COMPLETE — ALL RESULTS SAVED TO HF HUB")
print("="*55)
print("Model   : https://huggingface.co/Guuru-DEV/traffic-signal-agent")
print("Plots   : https://huggingface.co/datasets/Guuru-DEV/traffic-signal-plots")
print("WandB   : check wandb dashboard for reward curves")
print("="*55)
print("")
print("NEXT STEPS (do these locally after training):")
print("  1. Clone repo locally")
print("  2. Download plots from HF Hub:")
print("     huggingface-cli download Guuru-DEV/traffic-signal-plots --repo-type dataset --local-dir plots/")
print("  3. Add plots to README.md")
print("  4. git add plots/ README.md")
print("  5. git commit -m 'feat: real training results'")
print("  6. git push origin main")
print("  7. git push hf main")
print("")
print("PAUSE BOTH SPACES NOW TO STOP BILLING:")
print("  python scripts/pause_space.py")

